# 🗺️ Build Guide: Martin Tile Server

This guide walks through building Martin Tile Server from source and creating custom Docker images.

**Martin** is a high-performance Rust vector tile server that supports:
- PostgreSQL/PostGIS
- MBTiles files
- PMTiles files
- Sprite and font generation

---

## 📋 Table of Contents

1. [Prerequisites](#prerequisites)
2. [Direct Build](#direct-build)
3. [Installation from Cargo](#cargo-install)
4. [Development Commands](#development)
5. [Docker Image Creation](#docker)
6. [Verification](#verification)

## 🔧 Prerequisites

Before building Martin, ensure you have these tools installed:

### Required Tools:
- **Git** 2.30+ (version control - required to build maplibre_native)
- **Rust** 1.92+ (primary language)
- **Just** (task runner, similar to Make but simpler)
- **CMake** 3.10+ (build system - required to build maplibre_native)
- **Node.js** 18+ (required to build martin-ui frontend)
- **Docker** (optional, for images and test database)
- **PostgreSQL** with **PostGIS 3.0+** (optional, for database features)

### System Dependencies (Ubuntu/Debian):

**Base libraries:**
- **pkg-config** (required to find libraries during compilation)
- **libssl-dev** (OpenSSL development libraries)
- **sqlite3** (SQLite database tool)
- **cmake** (build system for C/C++)

**maplibre_native dependencies:**
- **libcurl4-openssl-dev** (HTTP client)
- **libjpeg-dev** (JPEG encoding/decoding)
- **libpng-dev** (PNG encoding/decoding)
- **libwebp-dev** (WebP encoding/decoding)
- **libx11-dev** (X11 window system)
- **libuv1-dev** (asynchronous I/O)
- **libicu-dev** (Unicode internationalization)
- **libglfw3-dev** (OpenGL/Vulkan window creation)
- **glslang-dev** (GLSL shader compiler)
- **glslang-tools** (shader compilation tools)
- **spirv-tools** (SPIR-V tools for Vulkan)
- **libspirv-cross-c-shared-dev** (SPIR-V shader conversion)

**Frontend:**
- **nodejs** and **npm** (JavaScript runtime and package manager)

```bash
# Install ALL dependencies on Ubuntu/Debian (full command):
sudo apt install -y pkg-config libssl-dev sqlite3 cmake \
  libcurl4-openssl-dev libjpeg-dev libpng-dev libwebp-dev \
  libx11-dev libuv1-dev libicu-dev libglfw3-dev \
  glslang-dev glslang-tools spirv-tools libspirv-cross-c-shared-dev

# Upgrade Git to a modern version (if needed):
sudo add-apt-repository ppa:git-core/ppa -y
sudo apt update
sudo apt install git -y

# Install Node.js 20 LTS (required for the frontend):
curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash -
sudo apt install -y nodejs
```

## ⚡ Quick Run (Steps 1-4)

If you're familiar with the build process, run all setup commands in one go:

In [ ]:
%%bash

# Step 0: Check/Upgrade Git
echo "🔍 Checking Git version..."
GIT_VERSION=$(git --version | grep -oP '\d+\.\d+' | head -1)
GIT_MAJOR=$(echo $GIT_VERSION | cut -d. -f1)
GIT_MINOR=$(echo $GIT_VERSION | cut -d. -f2)

if [ "$GIT_MAJOR" -lt 2 ] || ([ "$GIT_MAJOR" -eq 2 ] && [ "$GIT_MINOR" -lt 30 ]); then
    echo "⚠️ Git $GIT_VERSION is too old (required 2.30+)"
    echo "📥 Updating Git..."
    sudo add-apt-repository ppa:git-core/ppa -y
    sudo apt update
    sudo apt install git -y
    echo "✅ Git upgraded to: $(git --version)"
else
    echo "✅ Git is already up to date: $(git --version)"
fi

# Step 1: Check/Install Rust
if ! command -v rustc &> /dev/null; then
    echo "📥 Installing Rust..."
    curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
    source "$HOME/.cargo/env"
else
    echo "✅ Rust is already installed: $(rustc --version)"
fi

# Step 2: Install Just
if ! command -v just &> /dev/null; then
    echo "📥 Installing Just..."
    cargo install just --locked
else
    echo "✅ Just is already installed: $(just --version)"
fi

# Step 3: Install system dependencies (Ubuntu/Debian)
echo ""
echo "📦 Checking system dependencies..."

PACKAGES_TO_INSTALL=""

# Base libraries
if ! command -v sqlite3 &> /dev/null; then
    PACKAGES_TO_INSTALL="$PACKAGES_TO_INSTALL sqlite3"
fi

if ! command -v pkg-config &> /dev/null; then
    PACKAGES_TO_INSTALL="$PACKAGES_TO_INSTALL pkg-config"
fi

if ! dpkg -l | grep -q libssl-dev; then
    PACKAGES_TO_INSTALL="$PACKAGES_TO_INSTALL libssl-dev"
fi

if ! command -v cmake &> /dev/null; then
    PACKAGES_TO_INSTALL="$PACKAGES_TO_INSTALL cmake"
fi

# maplibre_native dependencies
for pkg in libcurl4-openssl-dev libjpeg-dev libpng-dev libwebp-dev libx11-dev libuv1-dev libicu-dev libglfw3-dev glslang-dev glslang-tools spirv-tools libspirv-cross-c-shared-dev; do
    if ! dpkg -l 2>/dev/null | grep -q "^ii.*$pkg"; then
        PACKAGES_TO_INSTALL="$PACKAGES_TO_INSTALL $pkg"
    fi
done

if [ -n "$PACKAGES_TO_INSTALL" ]; then
    echo "📥 Installing:$PACKAGES_TO_INSTALL"
    sudo apt install -y $PACKAGES_TO_INSTALL
else
    echo "✅ All system dependencies are already installed"
fi

# Step 3.5: Check/Install Node.js
echo ""
if ! command -v node &> /dev/null; then
    echo "📥 Installing Node.js 20 LTS..."
    curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash -
    sudo apt install -y nodejs
    echo "✅ Node.js installed: $(node --version)"
else
    NODE_VERSION=$(node --version | grep -oP '\d+' | head -1)
    if [ "$NODE_VERSION" -lt 18 ]; then
        echo "⚠️ Node.js $NODE_VERSION is too old (required 18+)"
        echo "📥 Actualizando Node.js..."
        curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash -
        sudo apt install -y nodejs
        echo "✅ Node.js upgraded: $(node --version)"
    else
        echo "✅ Node.js is already up to date: $(node --version)"
    fi
fi

# Step 4: Validate all tools
echo ""
echo "🔍 Validating tools..."
just validate-tools

echo ""
echo "✅ Setup complete! You can continue with the build."

---

**💡 Note:** For step-by-step details, continue reading. Otherwise, if the quick setup succeeded, skip to [Direct Build](#direct-build).

---

### Step 0: Check Git version

**IMPORTANT:** Git 2.30+ is required for `maplibre_native`. Older versions will cause build failures.

In [ ]:
%%bash
# Check current Git version
echo "🔍 Current Git version:"
git --version

# Check if version is 2.30 or higher
GIT_VERSION=$(git --version | grep -oP '\d+\.\d+' | head -1)
GIT_MAJOR=$(echo $GIT_VERSION | cut -d. -f1)
GIT_MINOR=$(echo $GIT_VERSION | cut -d. -f2)

if [ "$GIT_MAJOR" -lt 2 ] || ([ "$GIT_MAJOR" -eq 2 ] && [ "$GIT_MINOR" -lt 30 ]); then
    echo "⚠️ Git $GIT_VERSION is too old (required 2.30+)"
    echo "💡 Run the next cell to upgrade"
else
    echo "✅ Git is up to date (2.30+)"
fi

### Step 0.5: Update Git (if needed)

For Git versions older than 2.30, run this upgrade:

In [ ]:
%%bash
# Upgrade Git to the latest version using the official PPA
echo "📥 Adding official Git repository..."
sudo add-apt-repository ppa:git-core/ppa -y

echo "🔄 Updating package list..."
sudo apt update

echo "📦 Installing upgraded Git..."
sudo apt install git -y

echo ""
echo "✅ Git upgraded:"
git --version

---

### Step 1: Check if Rust is installed

In [ ]:
# Check Rust version
!rustc --version 2>/dev/null || echo "⚠️ Rust is not installed"

### Step 2: Install Rust (if not installed)

**⚠️ IMPORTANT:** This command downloads an installation script from the internet. Run it only if you trust the source.

In [ ]:
# Install Rust (uncomment to run)
# !curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh

# After installing, load the environment
# !source $HOME/.cargo/env

!echo "👉 Uncomment the lines above if you need to install Rust"

### Step 3: Install Just (task runner)

In [ ]:
# Install Just using Cargo
!cargo install just --locked

# Verify installation
!just --version

### Step 3.5: Install system dependencies

**Ubuntu/Debian only:** These system packages are required to compile Martin.

In [ ]:
%%bash
# Install ALL system dependencies required to build Martin
# This command installs everything at once

# Base libraries
# pkg-config: find libraries during compilation
# libssl-dev: OpenSSL development libraries
# sqlite3: SQLite database tool
# cmake: C/C++ build system

# maplibre_native dependencies (map rendering)
# libcurl4-openssl-dev: HTTP client
# libjpeg-dev: JPEG encoding/decoding
# libpng-dev: PNG encoding/decoding
# libwebp-dev: WebP encoding/decoding
# libx11-dev: X11 window system
# libuv1-dev: asynchronous I/O
# libicu-dev: Unicode internationalization
# libglfw3-dev: OpenGL/Vulkan window creation
# glslang-dev, glslang-tools: GLSL shader compiler
# spirv-tools: SPIR-V tools for Vulkan
# libspirv-cross-c-shared-dev: SPIR-V shader conversion

packages=(
  pkg-config libssl-dev sqlite3 cmake
  libcurl4-openssl-dev libjpeg-dev libpng-dev libwebp-dev
  libx11-dev libuv1-dev libicu-dev libglfw3-dev
  glslang-dev glslang-tools spirv-tools libspirv-cross-c-shared-dev
)

missing=()
for pkg in "${packages[@]}"; do
  if ! dpkg -s "$pkg" >/dev/null 2>&1; then
    missing+=("$pkg")
  fi
done

if [ ${#missing[@]} -eq 0 ]; then
  echo "[OK] All system dependencies are already installed"
  exit 0
fi

echo "[INFO] Missing packages: ${missing[*]}"

if [ "$EUID" -eq 0 ]; then
  apt-get update
  apt-get install -y "${missing[@]}"
  echo "[OK] Missing dependencies installed as root"
  exit 0
fi

if sudo -n true 2>/dev/null; then
  sudo apt install -y "${missing[@]}"
  echo "[OK] Missing dependencies installed with sudo"
  exit 0
fi

echo "[INFO] This notebook kernel cannot prompt for sudo password."
echo "Run this in terminal: sudo apt install -y ${missing[*]}"
echo "Then rerun this notebook cell."
exit 0

### Step 3.6: Install CMake

**Required to build maplibre_native:** CMake is the build system used by maplibre's C++ dependency.

In [ ]:
%%bash
# Install CMake (required to build maplibre_native)

if command -v cmake >/dev/null 2>&1; then
  echo "[OK] CMake already installed: $(cmake --version | head -n 1)"
  exit 0
fi

if [ "$EUID" -eq 0 ]; then
  apt-get update
  apt-get install -y cmake
  echo "[OK] CMake installed: $(cmake --version | head -n 1)"
  exit 0
fi

if sudo -n true 2>/dev/null; then
  sudo apt install -y cmake
  echo "[OK] CMake installed: $(cmake --version | head -n 1)"
  exit 0
fi

echo "[INFO] This notebook kernel cannot prompt for sudo password."
echo "Run this in terminal: sudo apt install -y cmake"
echo "Then rerun this notebook cell."
exit 0

### Step 3.7: Install Node.js 18+

**Required to build martin-ui:** Martin's frontend is written in TypeScript/React and requires Node.js to build.

In [ ]:
%%bash
# Check/install Node.js 20 LTS when needed

need_install=0
if command -v node >/dev/null 2>&1; then
  NODE_VERSION=$(node --version | sed 's/^v//' | cut -d. -f1)
  echo "[INFO] Current Node.js: $(node --version)"
  if [ "${NODE_VERSION:-0}" -lt 18 ]; then
    need_install=1
  fi
else
  need_install=1
fi

if [ "$need_install" -eq 0 ]; then
  echo "[OK] Node.js is compatible"
  npm --version
  exit 0
fi

install_cmd="curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash - && sudo apt install -y nodejs"

if [ "$EUID" -eq 0 ]; then
  curl -fsSL https://deb.nodesource.com/setup_20.x | bash -
  apt-get install -y nodejs
  echo "[OK] Node.js installed: $(node --version)"
  npm --version
  exit 0
fi

if sudo -n true 2>/dev/null; then
  eval "$install_cmd"
  echo "[OK] Node.js installed: $(node --version)"
  npm --version
  exit 0
fi

echo "[INFO] This notebook kernel cannot prompt for sudo password."
echo "Run this in terminal: $install_cmd"
echo "Then rerun this notebook cell."
exit 0

### Step 4: Validate all required tools

In [ ]:
# Validate that all tools are installed
!just validate-tools

---

## 🏗️ Direct Build

Use this approach for active development or when you need custom build configurations.

### Step 1: Start test database

Start PostgreSQL with PostGIS via Docker Compose:

In [33]:
import re
import subprocess


def run(cmd):
    return subprocess.run(cmd, capture_output=True, text=True)


def detect_busy_port(output: str):
    # Matches both formats seen in Docker errors:
    # - Bind for 0.0.0.0:5412 failed: port is already allocated
    # - Bind for :::5411 failed: port is already allocated
    m = re.search(r"Bind for [^:]+:(\d+) failed: port is already allocated", output)
    return m.group(1) if m else None


print("[INFO] Starting test services with 'just start'...")
first = run(["just", "start"])
combined = (first.stdout or "") + (first.stderr or "")
print(combined)

if first.returncode == 0:
    print("[OK] Test services are up")
else:
    busy_port = detect_busy_port(combined)

    if busy_port:
        print(f"[INFO] Port {busy_port} is busy. Cleaning local compose stack and retrying once...")
        down = run(["docker", "compose", "down", "--remove-orphans"])
        print((down.stdout or "") + (down.stderr or ""))

        retry = run(["just", "start"])
        retry_combined = (retry.stdout or "") + (retry.stderr or "")
        print(retry_combined)

        if retry.returncode == 0:
            print("[OK] Test services are up after retry")
        else:
            retry_busy_port = detect_busy_port(retry_combined) or busy_port
            print("[ERROR] Could not start services after retry")
            print(f"[INFO] Check who is using port {retry_busy_port}:")
            print(f"  docker ps --format '{{{{.ID}}}}\\t{{{{.Names}}}}\\t{{{{.Ports}}}}' --filter publish={retry_busy_port}")
            print(f"  ss -ltnp | grep :{retry_busy_port}")
            print("[INFO] Stop the conflicting service and rerun this cell.")
    else:
        print("[ERROR] 'just start' failed for a reason other than a Docker port conflict.")
        print("[INFO] Run this for details: just start")

[INFO] Starting test services with 'just start'...
localhost:5411 - accepting connections
Waiting for fileserver to be ready at http://localhost:5412...
Fileserver is ready!
docker compose up -d db
 Container martin-object_store-apikey-db-1 Starting 
 Container martin-object_store-apikey-db-1 Started 
docker compose run -T --rm db-is-ready
 Container martin-object_store-apikey-db-is-ready-run-81e3283e282c Creating 
 Container martin-object_store-apikey-db-is-ready-run-81e3283e282c Created 
docker compose up -d fileserver
 Container martin-object_store-apikey-fileserver-1 Starting 
 Container martin-object_store-apikey-fileserver-1 Started 

[OK] Test services are up


### Step 2: Build in Debug mode (faster)

**⏱️ Estimated time:** 5-15 minutes (first time)

**⚠️ IMPORTANT:** Per project guidelines, NEVER cancel `cargo build` once started.

In [ ]:
%%bash
# Build the entire workspace in debug mode
# Note: the `martin` crate builds `martin-ui` (npm) when `webui` feature is enabled.

if cargo build --workspace; then
  echo "[OK] Compiled binaries in: target/debug/"
  ls -lh target/debug/martin target/debug/martin-cp target/debug/mbtiles 2>/dev/null || echo "Build in progress..."
  exit 0
fi

echo ""
echo "[INFO] Build failed. Common cause: npm timeout while building martin-ui (webui)."
echo "[INFO] Retry frontend deps from terminal:"
echo "  cd martin/martin-ui && npm clean-install --no-fund --fetch-retries=5 --fetch-retry-mintimeout=20000 --fetch-retry-maxtimeout=120000"
echo "[INFO] Then rerun this cell."
echo ""
echo "[INFO] If you need a quick backend-only compile (without webui), use:"
echo "  cargo build -p martin --no-default-features --features fonts,lambda,mbtiles,metrics,pmtiles,postgres,rendering,sprites,styles,mlt"
exit 0

### Step 3: Build in Release mode (optimized)

**⏱️ Estimated time:** 10-30 minutes (first time)

Release binaries are **significantly faster** at runtime.

In [ ]:
%%bash
# Build in release mode (optimized)
# Note: the `martin` crate builds `martin-ui` (npm) when `webui` feature is enabled.

if cargo build --release --workspace; then
  echo "[OK] Optimized binaries in: target/release/"
  ls -lh target/release/martin target/release/martin-cp target/release/mbtiles 2>/dev/null || echo "Build in progress..."
  exit 0
fi

echo ""
echo "[INFO] Release build failed. Common cause: npm timeout while building martin-ui (webui)."
echo "[INFO] Retry frontend deps from terminal:"
echo "  cd martin/martin-ui && npm clean-install --no-fund --fetch-retries=5 --fetch-retry-mintimeout=20000 --fetch-retry-maxtimeout=120000"
echo "[INFO] Then rerun this cell."
echo ""
echo "[INFO] If you need a quick backend-only release compile (without webui), use:"
echo "  cargo build -p martin --release --no-default-features --features fonts,lambda,mbtiles,metrics,pmtiles,postgres,rendering,sprites,styles,mlt"
exit 0

### Verify the build

In [34]:
# Show help for compiled binaries
!target/release/martin --version
!target/release/martin-cp --help | head -n 10
!target/release/mbtiles --help | head -n 10

martin 1.10.0
A tool to bulk copy tiles from any Martin-supported sources into an mbtiles file

Usage: martin-cp [OPTIONS] --output-file <OUTPUT_FILE> [CONNECTION]...

Arguments:
  [CONNECTION]...
          Connection strings, e.g. postgres://... or /path/to/files

Options:
  -s, --source <SOURCE>
A utility to work with .mbtiles file content

Usage: mbtiles <COMMAND>

Commands:
  summary      Show MBTiles file summary statistics
  meta-all     Prints all values in the metadata table in a free-style, unstable
               YAML format
  meta-get     Gets a single value from the MBTiles metadata table
  meta-set     Sets a single value in the MBTiles metadata table or deletes it


---

## 📦 Installation from Cargo

Use this if you just want to run Martin without modifying the source:

In [ ]:
# Install Martin from crates.io (published version)
# Builds from source without requiring a local repository clone
# !cargo install martin --locked

# Verify installation
# !martin --version

!echo "👉 Uncomment the lines above to install from cargo"

---

## 🚀 Useful Development Commands

Helpful commands for Martin development:

### View all available commands

In [30]:
import subprocess

# List all available Just commands without piping to head
result = subprocess.run(["just", "--list"], capture_output=True, text=True)

if result.returncode != 0:
    print("[ERROR] Failed to run 'just --list'")
    print(result.stderr.strip())
else:
    lines = result.stdout.splitlines()
    print("\n".join(lines[:30]))

Available recipes:
    bench                                        # Run benchmark tests
    bench-http requests='10m' pg_requests='500k' # Run HTTP requests benchmark using OHA tool. Use with `just bench-server`.
    bench-server                                 # Start release-compiled Martin server and a test database
    bench-server-hotpath                         # Start release-compiled Martin server with hotpath profiling (MCP on port 6771)
    bless                                        # Run integration tests and save its output as the new expected output (ordering is important)
    bless-insta *args                            # Run integration tests and save its output as the new expected output
    bless-int                                    # Bless integration tests
    bless-pg
    build-deb output                             # and maplibre_native pre-built libraries require newer glibc.
    build-hotpath                                # Build martin with hotpath profil

### Run the server (development)

In [ ]:
# Run Martin in development mode (with logging)
# !just run

# Or run directly with test MBTiles/PMTiles files
# !cargo run --bin martin -- tests/fixtures/mbtiles tests/fixtures/pmtiles

# Or run the release version
# !cargo run --release --bin martin -- tests/fixtures/mbtiles

!echo "👉 Uncomment one of the commands above to run the server"
!echo "🌐 Server will be available at http://localhost:3000"

### Run tests

In [ ]:
# Run tests unitarios
# !cargo test --workspace

# Or use just to run all tests (including integration)
# !just test

!echo "⚠️ Tests may take several minutes"
!echo "👉 Uncomment to run tests"

### Check code without building binaries (faster)

In [ ]:
# Verify code compiles without generating binaries
# !just check

# Or use clippy for linting
# !cargo clippy --workspace

!echo "👉 Uncomment to run checks"

### Format code

In [ ]:
# Format all Rust code
# !just fmt

!echo "👉 Uncomment to format the code"

---

## 📚 Additional Resources

### Official Documentation
- **Sitio web:** https://maplibre.org/martin/
- **GitHub:** https://github.com/maplibre/martin
- **Quick Start:** https://maplibre.org/martin/quick-start.html

### Useful Project Commands

```bash
just help              # Help for new contributors
just validate-tools    # Validate required tools
just start             # Start test database
just run               # Start Martin server
just test              # Run all tests
just fmt               # Format code
just docs              # Serve preview documentation
just --list            # View all available commands
```

### Important Project Guidelines

🔧 **Version requirements:**
- Git 2.30+ is required to build maplibre_native
- Rust 1.92+ es la minimum supported version
- CMake 3.10+ is required to build maplibre_native
- Node.js 18+ is required to build martin-ui

📦 **Critical system dependencies (Ubuntu/Debian):**
- **Base:** pkg-config, libssl-dev, sqlite3, cmake
- **maplibre_native:** libcurl4-openssl-dev, libjpeg-dev, libpng-dev, libwebp-dev, libx11-dev, libuv1-dev, libicu-dev, libglfw3-dev, glslang-dev, glslang-tools, spirv-tools, libspirv-cross-c-shared-dev
- **Frontend:** nodejs, npm

💡 **Full installation command:**
```bash
sudo apt install -y pkg-config libssl-dev sqlite3 cmake \
  libcurl4-openssl-dev libjpeg-dev libpng-dev libwebp-dev \
  libx11-dev libuv1-dev libicu-dev libglfw3-dev \
  glslang-dev glslang-tools spirv-tools libspirv-cross-c-shared-dev
```

⚠️ **NEVER cancel** `cargo build`, `cargo test`, `just check`, o `just ci-test` once started

⏱️ **Recommended timeouts:**
- `cargo build --workspace`: 20 minutes
- `just test`: 30 minutes
- `just ci-test`: 45 minutes

---

## 🎉 Done!

You now have a complete guide to build Martin Tile Server and create Docker images.

You can run the notebook cells step by step to follow the build process.